# Task 3: Tool for String Manipulation

This notebook demonstrates an agent that uses a tool to reverse strings.

## 1. Define the String Reversal Tool

First, we'll create a tool class that can reverse strings.

In [ ]:
from typing import Any, Dict


class StringReversalTool:
    """A tool that reverses strings."""
    
    name = "reverse_string"
    description = "Reverses the characters in a given string"
    
    def execute(self, input_string: str) -> str:
        """Reverse the input string.
        
        Args:
            input_string: The string to reverse
            
        Returns:
            The reversed string
        """
        return input_string[::-1]
    
    def get_schema(self) -> Dict[str, Any]:
        """Return the tool schema."""
        return {
            "name": self.name,
            "description": self.description,
            "parameters": {
                "type": "object",
                "properties": {
                    "input_string": {
                        "type": "string",
                        "description": "The string to reverse"
                    }
                },
                "required": ["input_string"]
            }
        }


# Test the tool directly
tool = StringReversalTool()
print(f"Tool name: {tool.name}")
print(f"Tool description: {tool.description}")
print(f"Test: 'hello' -> '{tool.execute('hello')}'")

## 2. Create the String Reversal Agent

Now we'll create an agent that can use the string reversal tool to process requests.

In [ ]:
import re
from typing import List, Optional


class StringReversalAgent:
    """An agent that uses tools to manipulate strings."""
    
    def __init__(self):
        self.tools = {
            "reverse_string": StringReversalTool()
        }
        self.history: List[Dict[str, str]] = []
    
    def _parse_request(self, user_input: str) -> Optional[str]:
        """Extract the string to reverse from user input."""
        # Look for quoted strings
        quoted = re.findall(r'["\']([^"\']+)["\']', user_input)
        if quoted:
            return quoted[0]
        
        # Look for "reverse X" patterns
        match = re.search(r'reverse\s+(.+)', user_input.lower())
        if match:
            return match.group(1).strip()
        
        return None
    
    def _decide_action(self, user_input: str) -> Dict[str, Any]:
        """Decide which tool to use based on user input."""
        user_lower = user_input.lower()
        
        # Check if user wants to reverse a string
        if any(word in user_lower for word in ["reverse", "flip", "backward"]):
            string_to_reverse = self._parse_request(user_input)
            if string_to_reverse:
                return {
                    "tool": "reverse_string",
                    "params": {"input_string": string_to_reverse}
                }
        
        return {"tool": None, "params": {}}
    
    def _execute_tool(self, tool_name: str, params: Dict[str, Any]) -> str:
        """Execute the specified tool with given parameters."""
        if tool_name not in self.tools:
            return f"Error: Unknown tool '{tool_name}'"
        
        tool = self.tools[tool_name]
        return tool.execute(**params)
    
    def run(self, user_input: str) -> str:
        """Process user input and return a response."""
        # Store in history
        self.history.append({"role": "user", "content": user_input})
        
        # Decide action
        action = self._decide_action(user_input)
        
        if action["tool"]:
            # Execute tool
            result = self._execute_tool(action["tool"], action["params"])
            response = f"Using tool: {action['tool']}\nInput: {action['params']['input_string']}\nResult: {result}"
        else:
            response = "I can help you reverse strings! Try asking me to 'reverse \"your text\"'"
        
        # Store response in history
        self.history.append({"role": "assistant", "content": response})
        
        return response


# Create agent instance
agent = StringReversalAgent()
print("String Reversal Agent initialized!")
print(f"Available tools: {list(agent.tools.keys())}")

## 3. Demonstrate the Agent

Let's test the agent with various inputs.

In [ ]:
# Test 1: Simple string reversal
response = agent.run('Please reverse "Hello World"')
print(response)

In [ ]:
# Test 2: Another reversal request
response = agent.run('Can you flip "Python is great"')
print(response)

In [ ]:
# Test 3: Palindrome check (reversed string equals original)
response = agent.run('reverse "racecar"')
print(response)

In [ ]:
# Test 4: Unknown request (agent provides help)
response = agent.run('What can you do?')
print(response)

## 4. Batch Processing

Process multiple strings at once.

In [ ]:
# Batch reverse multiple strings
strings_to_reverse = [
    "OpenAI",
    "Machine Learning",
    "Artificial Intelligence",
    "12345"
]

print("Batch String Reversal:")
print("-" * 50)

for s in strings_to_reverse:
    response = agent.run(f'reverse "{s}"')
    print(response)
    print("-" * 50)

## 5. View Agent History

The agent maintains a conversation history.

In [ ]:
# Display conversation history
print("Agent Conversation History:")
print("=" * 60)

for i, entry in enumerate(agent.history):
    role = entry["role"].upper()
    content = entry["content"]
    print(f"[{i+1}] {role}:")
    print(f"    {content[:100]}..." if len(content) > 100 else f"    {content}")
    print()

## Summary

This notebook demonstrated:

1. **Tool Definition**: Created a `StringReversalTool` class with a schema and execute method
2. **Agent Creation**: Built a `StringReversalAgent` that can parse requests, decide actions, and execute tools
3. **Interactive Usage**: Showed how to interact with the agent using natural language
4. **Batch Processing**: Demonstrated processing multiple strings
5. **History Tracking**: The agent maintains conversation history for context